# Reading files as a stream

In [1]:
import earthkit.data as ekd

ekd.download_example_file("test6.grib")

### Getting single items from the stream

In [2]:
ds_in = ekd.from_source("file", "test6.grib", stream=True)
ds_in

types,fieldlist


To access the stream data we need to convert the data object into a stream fieldlist.

In [3]:
ds = ds_in.to_fieldlist()

In [4]:
for f in ds:
    # f is GribField object. It gets deleted when going out of scope
    print(f)

Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(u, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(u, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)


Having finished the iteration there is no data available in *ds*. 

In [5]:
len([f in ds])

1

### Using batched

In [6]:
ds = ekd.from_source("file", "test6.grib", stream=True).to_fieldlist()

# f is a fieldlist
for f in ds.batched(2):
    print(f"len={len(f)} {f.get(['parameter.variable', 'vertical.level'])}")

len=2 [['t', 1000], ['u', 1000]]
len=2 [['v', 1000], ['t', 850]]
len=2 [['u', 850], ['v', 850]]


In [7]:
ds = ekd.from_source("file", "test6.grib", stream=True).to_fieldlist()

# f is a fieldlist
for f in ds.batched(4):
    print(f"len={len(f)} {f.get(['parameter.variable', 'vertical.level'])}")

len=4 [['t', 1000], ['u', 1000], ['v', 1000], ['t', 850]]
len=2 [['u', 850], ['v', 850]]


### Using group_by

In [8]:
ds = ekd.from_source("file", "test6.grib", stream=True).to_fieldlist()

# f is a fieldlist
for f in ds.group_by("level"):
    print(f"len={len(f)} {f.get(['parameter.variable', 'vertical.level'])}")

len=6 [['t', 1000], ['u', 1000], ['v', 1000], ['t', 850], ['u', 850], ['v', 850]]


### Storing each GRIB message in memory

In [9]:
ds = ekd.from_source("file", "test6.grib", stream=True).to_fieldlist(read_all=True)

In [10]:
len(ds)

6

In [11]:
ds.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
2,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
3,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
4,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
5,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


In [12]:
a = ds.sel({"parameter.variable": "t"})
a.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


In [13]:
a = a.to_xarray()
a

<xarray.Dataset> Size: 2kB
Dimensions:    (level: 2, latitude: 7, longitude: 12)
Coordinates:
  * level      (level) int64 16B 850 1000
  * latitude   (latitude) float64 56B 90.0 60.0 30.0 0.0 -30.0 -60.0 -90.0
  * longitude  (longitude) float64 96B 0.0 30.0 60.0 90.0 ... 270.0 300.0 330.0
Data variables:
    t          (level, latitude, longitude) float64 1kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF